# Lakebase Support App — Console

Runs directly against the **Lakebase Postgres** instance behind the support ticket app,
to verify the schema, the sample data, and that reads and writes actually work.

**Why a notebook and not a `.sql` file?** A `.sql` file in the Databricks editor executes
on a SQL warehouse against Unity Catalog, which cannot reach Lakebase Postgres at all.
This notebook uses `psycopg2` in Python cells to connect to the Postgres endpoint natively.

**Prerequisites**

| Thing | Value |
|---|---|
| Secret scope / key | `support` / `lakebase-url` |
| Postgres role | `support_app` |
| Database | `databricks_postgres` |
| Tables | `tickets`, `ticket_messages` |

If the secret scope does not exist yet, run `setup_secrets.py` from the repo first.

Run the cells in order. Section 6 writes a test ticket and deletes it again, so nothing
here changes your sample data.

In [0]:
import warnings

import psycopg2
import pandas as pd

# pandas warns once per read_sql call that a raw DBAPI connection is not a
# SQLAlchemy connectable. It works fine against psycopg2, and the alternative -
# pulling in SQLAlchemy just to silence it - is not worth it here. Suppressing
# only this message keeps the notebook output readable.
warnings.filterwarnings("ignore", message=".*only supports SQLAlchemy connectable.*")

# dbutils.secrets.get() returns the DECODED value, unlike the Databricks SDK's
# secrets API which returns base64. Falling back to the env var lets this same
# notebook run locally (VS Code, Jupyter) where dbutils does not exist.
try:
    LAKEBASE_URL = dbutils.secrets.get(scope="support", key="lakebase-url")
    print("Read the connection URL from secret scope 'support'.")
except NameError:
    import os
    LAKEBASE_URL = os.environ["LAKEBASE_URL"]
    print("No dbutils here - read the connection URL from $LAKEBASE_URL.")

def connect(url):
    """Open a Lakebase connection, tolerating a percent-encoded password.

    Lakebase hands you the URL with the password percent-encoded. psycopg2
    usually copes, but a password containing '/' or '@' breaks naive parsing,
    so fall back to passing the components explicitly.
    """
    try:
        return psycopg2.connect(url)
    except psycopg2.Error:
        from urllib.parse import urlparse, unquote
        p = urlparse(url)
        return psycopg2.connect(
            host=p.hostname, port=p.port or 5432,
            dbname=p.path.lstrip("/") or "databricks_postgres",
            user=unquote(p.username), password=unquote(p.password),
            sslmode="require",
        )

conn = connect(LAKEBASE_URL)

# display() only exists in Databricks; print a plain table when running elsewhere.
try:
    display
except NameError:
    display = lambda df: print(df.to_string(index=False))

def q(sql, params=None):
    """Run a query and return a DataFrame."""
    return pd.read_sql_query(sql, conn, params=params)

who = q("SELECT current_user AS role, current_database() AS database, version() AS version")
print(f"Connected as {who.role[0]} to {who.database[0]}")
print(who.version[0].split(",")[0])

---
## 1. Schema — the two required tables

The assignment requires two related tables. `tickets` carries extra columns for the bonus
challenges (`description`, `priority`, `category`, `updated_at`), which is explicitly allowed.

In [0]:
tables = q("""
    SELECT table_name,
           (SELECT COUNT(*) FROM information_schema.columns c
            WHERE c.table_schema = t.table_schema AND c.table_name = t.table_name) AS columns
    FROM information_schema.tables t
    WHERE table_schema = 'public' AND table_type = 'BASE TABLE'
    ORDER BY table_name
""")
print("=== Tables in schema public ===")
display(tables)

REQUIRED = {
    "tickets":         ["ticket_id", "title", "status", "created_by", "created_at"],
    "ticket_messages": ["message_id", "ticket_id", "message_text", "author", "created_at"],
}

cols = q("""
    SELECT table_name, ordinal_position, column_name, data_type, is_nullable, column_default
    FROM information_schema.columns
    WHERE table_schema = 'public' AND table_name IN ('tickets', 'ticket_messages')
    ORDER BY table_name, ordinal_position
""")
cols["required_by_assignment"] = [
    c in REQUIRED.get(t, []) for t, c in zip(cols.table_name, cols.column_name)
]
print("=== Columns ===")
display(cols)

for table, required in REQUIRED.items():
    have = set(cols[cols.table_name == table].column_name)
    missing = [c for c in required if c not in have]
    print(f"{table}: {'ALL REQUIRED COLUMNS PRESENT' if not missing else 'MISSING ' + ', '.join(missing)}")

In [0]:
fk = q("""
    SELECT tc.constraint_name,
           kcu.column_name  AS fk_column,
           ccu.table_name   AS references_table,
           ccu.column_name  AS references_column,
           rc.delete_rule
    FROM information_schema.table_constraints tc
    JOIN information_schema.key_column_usage kcu
      ON kcu.constraint_name = tc.constraint_name
    JOIN information_schema.constraint_column_usage ccu
      ON ccu.constraint_name = tc.constraint_name
    JOIN information_schema.referential_constraints rc
      ON rc.constraint_name = tc.constraint_name
    WHERE tc.constraint_type = 'FOREIGN KEY' AND tc.table_name = 'ticket_messages'
""")
print("=== Foreign key ===")
display(fk)

# Prove it is enforced, not merely declared. The rollback leaves no trace.
try:
    with conn.cursor() as cur:
        cur.execute("INSERT INTO ticket_messages (ticket_id, message_text, author) "
                    "VALUES (99999999, 'orphan probe', 'notebook')")
    print("PROBLEM: an orphaned message was accepted")
except psycopg2.errors.ForeignKeyViolation:
    print("ENFORCED: Postgres rejected a message pointing at a nonexistent ticket")
finally:
    conn.rollback()

orphans = q("""
    SELECT COUNT(*) AS orphaned_messages
    FROM ticket_messages m
    LEFT JOIN tickets t ON t.ticket_id = m.ticket_id
    WHERE t.ticket_id IS NULL
""")
print(f"Orphaned messages: {int(orphans.orphaned_messages[0])} (expected 0)")

In [0]:
checks = q("""
    SELECT conname AS constraint_name,
           pg_get_constraintdef(oid) AS definition
    FROM pg_constraint
    WHERE conrelid = 'tickets'::regclass AND contype = 'c'
    ORDER BY conname
""")
display(checks)

try:
    with conn.cursor() as cur:
        cur.execute("INSERT INTO tickets (title, status, created_by) VALUES ('probe', 'banana', 'notebook')")
    print("PROBLEM: an invalid status was accepted")
except psycopg2.errors.CheckViolation:
    print("ENFORCED: Postgres rejected status = 'banana'")
finally:
    conn.rollback()

---
## 2. Sample data requirements

The assignment asks for at least three tickets, at least two messages per ticket, and at
least two distinct statuses.

In [0]:
summary = q("""
    SELECT t.ticket_id, t.title, t.status, t.priority, t.category,
           t.created_by, t.created_at,
           COUNT(m.message_id) AS message_count
    FROM tickets t
    LEFT JOIN ticket_messages m ON m.ticket_id = t.ticket_id
    GROUP BY t.ticket_id, t.title, t.status, t.priority, t.category, t.created_by, t.created_at
    ORDER BY t.ticket_id
""")
display(summary)

n_tickets  = len(summary)
n_statuses = summary.status.nunique()
thin       = summary[summary.message_count < 2]

results = [
    (n_tickets >= 3,   f"at least 3 tickets (have {n_tickets})"),
    (n_statuses >= 2,  f"at least 2 distinct statuses (have {n_statuses}: {', '.join(sorted(summary.status.unique()))})"),
    (thin.empty,       "at least 2 messages per ticket" + ("" if thin.empty else f" - short: {list(thin.ticket_id)}")),
]
print()
for ok, label in results:
    print(f"[{'PASS' if ok else 'FAIL'}] {label}")

---
## 3. Browse the data

The same rows the app serves — this is where the app's ticket list and message thread
come from.

In [0]:
display(q("""
    SELECT t.ticket_id, t.title, t.status, t.priority, t.category,
           t.created_by,
           COUNT(m.message_id)                       AS messages,
           COALESCE(MAX(m.created_at), t.created_at) AS last_activity
    FROM tickets t
    LEFT JOIN ticket_messages m ON m.ticket_id = t.ticket_id
    GROUP BY t.ticket_id
    ORDER BY t.updated_at DESC
"""))

In [0]:
# Change this to inspect a different ticket.
TICKET_ID = int(q("SELECT MIN(ticket_id) AS id FROM tickets").id[0])

ticket = q("SELECT * FROM tickets WHERE ticket_id = %(id)s", {"id": TICKET_ID})
print(f"=== Ticket #{TICKET_ID}: {ticket.title[0]} ===")
print(f"status={ticket.status[0]}  priority={ticket.priority[0]}  category={ticket.category[0]}")
print(f"opened by {ticket.created_by[0]} at {ticket.created_at[0]}\n")

display(q("""
    SELECT message_id, author, created_at, message_text
    FROM ticket_messages
    WHERE ticket_id = %(id)s
    ORDER BY created_at, message_id
""", {"id": TICKET_ID}))

---
## 4. Statistics

The same aggregates the app's `/api/stats` endpoint returns and renders as tiles.

In [0]:
display(q("""
    SELECT COUNT(*)                                       AS total_tickets,
           COUNT(*) FILTER (WHERE status = 'open')        AS open_tickets,
           COUNT(*) FILTER (WHERE status = 'in_progress') AS in_progress,
           COUNT(*) FILTER (WHERE status = 'resolved')    AS resolved,
           COUNT(*) FILTER (WHERE status = 'closed')      AS closed,
           COUNT(*) FILTER (WHERE priority IN ('high','urgent')
                              AND status NOT IN ('resolved','closed')) AS urgent_open,
           (SELECT COUNT(*) FROM ticket_messages)         AS total_messages,
           ROUND((SELECT COUNT(*) FROM ticket_messages)::numeric
                 / NULLIF(COUNT(*), 0), 1)                AS avg_messages_per_ticket
    FROM tickets
"""))

for dimension in ("status", "priority", "category"):
    print(f"=== Tickets by {dimension} ===")
    display(q(f"SELECT {dimension}, COUNT(*) AS count FROM tickets "
              f"GROUP BY {dimension} ORDER BY count DESC"))

In [0]:
print("=== Tickets with the most messages ===")
display(q("""
    SELECT t.ticket_id, t.title, t.status, COUNT(m.message_id) AS messages
    FROM tickets t
    JOIN ticket_messages m ON m.ticket_id = t.ticket_id
    GROUP BY t.ticket_id, t.title, t.status
    ORDER BY messages DESC, t.ticket_id
    LIMIT 10
"""))

print("=== Who is writing ===")
display(q("""
    SELECT author, COUNT(*) AS messages, COUNT(DISTINCT ticket_id) AS tickets
    FROM ticket_messages
    GROUP BY author
    ORDER BY messages DESC
"""))

---
## 5. Write test

Proves the database accepts writes from outside the app, and that `ON DELETE CASCADE`
really removes a ticket's messages.

Everything created here is deleted at the end of the cell — your sample data is untouched.

In [0]:
import uuid as _uuid
MARKER = f"[notebook-{_uuid.uuid4().hex[:8]}]"
before = int(q("SELECT COUNT(*) AS n FROM tickets").n[0])

with conn.cursor() as cur:
    cur.execute("""
        INSERT INTO tickets (title, description, status, priority, category, created_by)
        VALUES (%s, %s, 'open', 'medium', 'general', %s)
        RETURNING ticket_id
    """, (f"{MARKER} Notebook write test", "Created by the console notebook.", "notebook@test"))
    new_id = cur.fetchone()[0]

    cur.executemany("""
        INSERT INTO ticket_messages (ticket_id, message_text, author) VALUES (%s, %s, %s)
    """, [(new_id, "First message from the notebook.", "notebook@test"),
          (new_id, "Second message from the notebook.", "support@example.com")])
conn.commit()
print(f"Created ticket #{new_id} with 2 messages")

display(q("""
    SELECT t.ticket_id, t.title, t.status, COUNT(m.message_id) AS messages
    FROM tickets t LEFT JOIN ticket_messages m ON m.ticket_id = t.ticket_id
    WHERE t.ticket_id = %(id)s GROUP BY t.ticket_id, t.title, t.status
""", {"id": new_id}))

# Update, to show a status change persists.
with conn.cursor() as cur:
    cur.execute("UPDATE tickets SET status = 'resolved', updated_at = now() WHERE ticket_id = %s", (new_id,))
conn.commit()
status = q("SELECT status FROM tickets WHERE ticket_id = %(id)s", {"id": new_id}).status[0]
print(f"After update, status = {status}")

# Delete the ticket; its messages should go with it.
msgs_before = int(q("SELECT COUNT(*) AS n FROM ticket_messages WHERE ticket_id = %(id)s",
                    {"id": new_id}).n[0])
with conn.cursor() as cur:
    cur.execute("DELETE FROM tickets WHERE ticket_id = %s", (new_id,))
conn.commit()
msgs_after = int(q("SELECT COUNT(*) AS n FROM ticket_messages WHERE ticket_id = %(id)s",
                   {"id": new_id}).n[0])
after = int(q("SELECT COUNT(*) AS n FROM tickets").n[0])

print(f"\n[{'PASS' if status == 'resolved' else 'FAIL'}] status update persisted")
print(f"[{'PASS' if msgs_before == 2 and msgs_after == 0 else 'FAIL'}] "
      f"CASCADE removed the messages ({msgs_before} -> {msgs_after})")
print(f"[{'PASS' if after == before else 'FAIL'}] sample data unchanged ({before} tickets before, {after} after)")

---
## 6. Assignment checklist

One cell that answers every schema and data requirement at once — useful for a screenshot.

In [0]:
tables_present = set(q("""
    SELECT table_name FROM information_schema.tables
    WHERE table_schema='public' AND table_type='BASE TABLE'
""").table_name)

cols_now = q("""
    SELECT table_name, column_name FROM information_schema.columns
    WHERE table_schema='public' AND table_name IN ('tickets','ticket_messages')
""")
have = {t: set(cols_now[cols_now.table_name == t].column_name) for t in REQUIRED}

fk_ok = not q("""
    SELECT 1 FROM information_schema.table_constraints
    WHERE constraint_type='FOREIGN KEY' AND table_name='ticket_messages'
""").empty

data = q("""
    SELECT t.ticket_id, t.status, COUNT(m.message_id) AS messages
    FROM tickets t LEFT JOIN ticket_messages m ON m.ticket_id = t.ticket_id
    GROUP BY t.ticket_id, t.status
""")

checks = [
    ("tickets table exists",                 "tickets" in tables_present),
    ("ticket_messages table exists",         "ticket_messages" in tables_present),
    ("tickets has all required columns",     not set(REQUIRED["tickets"]) - have["tickets"]),
    ("ticket_messages has all required cols", not set(REQUIRED["ticket_messages"]) - have["ticket_messages"]),
    ("ticket_messages.ticket_id references tickets", fk_ok),
    ("at least 3 tickets",                   len(data) >= 3),
    ("at least 2 distinct statuses",         data.status.nunique() >= 2),
    ("at least 2 messages per ticket",       bool((data.messages >= 2).all())),
]

report = pd.DataFrame(
    [{"requirement": label, "result": "PASS" if ok else "FAIL"} for label, ok in checks])
display(report)

failed = [label for label, ok in checks if not ok]
print("\nALL REQUIREMENTS MET" if not failed else "NOT MET: " + "; ".join(failed))

In [0]:
conn.close()
print("Connection closed.")

---
### Notes

* **Do not convert these cells to `%sql`.** A `%sql` cell runs on Unity Catalog, which is a
  different engine and cannot see these tables. Everything here goes through `psycopg2` on
  purpose.
* If the connection fails with `password authentication failed`, suspect the **host** before
  the password — a Lakebase host already ends in `.database.<region>.cloud.databricks.com`,
  and appending anything to it produces a DNS failure that psycopg2 reports as an auth error.
* If a write fails with `permission denied for schema public`, run `sql/00_grant_app_role.sql`.